# Adaptive Large Neighborhood Search (ALNS) for 1D Bin Packing

This notebook is a self-contained explanation and minimal execution workflow for the `hybrid_alns` solver in this repository.

---

## 1) Problem definition from first principles

### 1.1 Inputs
A **1D Bin Packing** instance gives:
- A multiset of item sizes: $s_1, s_2, \dots, s_n$ with $s_i \in \mathbb{Z}_{>0}$.
- A bin capacity: $C \in \mathbb{Z}_{>0}$.

### 1.2 Feasible solution
A solution is a partition of items into bins $B_1,\dots,B_m$ such that:
1. Every item appears exactly once: $\biguplus_{j=1}^{m} B_j = \{1,\dots,n\}$.
2. Capacity respected for every bin: $\sum_{i\in B_j} s_i \le C$.

### 1.3 Objective
Minimize the number of bins used:
\[
\min m.
\]

A standard lower bound is:
\[
\text{LB} = \left\lceil \frac{\sum_{i=1}^n s_i}{C} \right\rceil,
\]
which is necessary but usually not sufficient for optimality.

---

## 2) Why metaheuristics are useful

1D Bin Packing is NP-hard. Exact methods scale poorly on larger instances. Greedy heuristics (FFD/BFD) are fast but may be suboptimal. Local search improves solutions by modifications, but pure small-move neighborhoods often get trapped in local minima.

The core challenge is: **we need both intensification (improve current structure) and diversification (escape local minima)**.

---

## 3) LNS and ALNS: core idea

## 3.1 Large Neighborhood Search (LNS)
Instead of tiny edits, LNS repeatedly:
1. **Destroy**: remove part of current assignment.
2. **Repair**: rebuild that part, potentially in a better way.

Destroying many assignments creates a large neighborhood implicitly (all reconstructions reachable by repair), making escape from local minima much easier than one-item moves.

## 3.2 Adaptive LNS (ALNS)
ALNS keeps multiple destroy/repair operators and **adapts** which ones are used based on past performance.

In this repository’s hybrid solver:
- Destroy operators are multiple (random, worst, related).
- Repair is learned (ML-guided feasible-bin ranking).
- Destroy-operator selection is adaptive via Thompson Sampling.

---

## 4) Acceptance criterion: simulated annealing

Let objective be $f(x)=\#$bins. For current solution $x$ and candidate $x'$:
\[
\Delta = f(x') - f(x).
\]

- If $\Delta \le 0$, accept (non-worse).
- If $\Delta > 0$, accept with probability:
\[
P(\text{accept}) = \exp\left(-\frac{\Delta}{T}\right).
\]

Here $T$ is temperature. With cooling schedule (e.g., $T \leftarrow \alpha T$ with $\alpha<1$):
- Early iterations: more uphill acceptance (exploration).
- Later iterations: more greedy behavior (exploitation).

This prevents premature convergence.

---

## 5) Adaptive operator selection: Thompson Sampling

Treat each destroy operator as a bandit arm with unknown success probability.
For each arm $k$, maintain Beta posterior:
\[
\theta_k \sim \text{Beta}(\alpha_k,\beta_k).
\]
Per iteration:
1. Sample one value from each posterior.
2. Choose arm with highest sampled value.
3. Observe binary reward (e.g., useful accepted improvement vs not).
4. Update $(\alpha_k,\beta_k)$.

Why this works:
- Naturally balances exploration/exploitation.
- Arms with uncertainty still get sampled.
- Arms with repeated success become preferred.

---

## 6) Learned repair model

During repair, for each displaced item, several existing bins may be feasible. The decision is: **which feasible bin should receive the item?**

The hybrid approach frames this as supervised ranking/classification:
- Build features for each (item, feasible-bin) pair (item size, load, slack after placement, relative ranks/context, etc.).
- Train logistic regression to estimate suitability score/probability.
- In repair, evaluate feasible bins and place item in highest-scoring one.
- If none feasible, open a new bin.

Training labels are generated by replaying BFD behavior:
- Positive: the bin BFD selected.
- Negative: other feasible bins.

So the model imitates a strong constructive policy while ALNS search still provides global exploration through destroy/repair cycles.

---

## 7) End-to-end workflow used here

1. **Train** the repair model once (`train_repair_model.py`) and save it as `repair_model.pkl`.
2. **Benchmark** the hybrid solver (`benchmark.py`) and pass model path via `--method-args`.
3. Read the printed benchmark table/results.

No plotting is needed for this minimal workflow.


In [ ]:
!python train_repair_model.py \
  --instances 5000 \
  --n-min 50 \
  --n-max 200 \
  --max-negatives 5 \
  --seed 0 \
  --output repair_model.pkl

In [ ]:
!python ../../benchmark.py \
  --solver 5_hybrid_ml_metaheuristics/hybrid_alns/solver.py \
  --method hybrid_alns \
  --method-args model_path=5_hybrid_ml_metaheuristics/hybrid_alns/repair_model.pkl,max_iterations=5000 \
  --datasets falkenauer-t \
  --limit 5